# Step 3: Hi-C eigenvector centrality

Per-chromosome, per-stage eigenvector centrality on long-range (>=500kb), O/E-weighted contacts. Centrality is computed on each chromosome's largest connected component; bins outside it are NaN.

## Config

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse.linalg import eigsh
from scipy.sparse.csgraph import connected_components

ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"

HIC_DIR = RAW / "hic_contacts"
GENE_COORDS = PROC / "gene_coordinates.csv"
OUT = PROC / "eigenvector_features.csv"

STAGES = ["C1-8", "C9-13", "C14E", "C14L"]
MAIN_CHROMS = ["chr2L", "chr2R", "chr3L", "chr3R", "chr4", "chrX"]
BIN_SIZE = 10_000
MIN_DISTANCE_BINS = 500_000 // BIN_SIZE  # 50
EIG_MAX_ITER = 5000
EIG_TOL = 1e-6

assert GENE_COORDS.exists(), "run 01_parse_gtf first"
genes = pd.read_csv(GENE_COORDS)
print(f"{len(genes)} genes loaded")

16846 genes loaded


## O/E weights

In [2]:
# expected[d] = total observed count at distance d / total possible bin pairs at distance d
# (n - d pairs, assuming bin ids are contiguous per chromosome — checked below)

def oe_weights(df, bins_df, chrom):
    bin_ids = np.sort(bins_df.loc[bins_df["chr"] == chrom, "cbin"].unique())
    contiguous = np.all(np.diff(bin_ids) == 1)
    if not contiguous:
        print(f"  [warn] {chrom}: bin ids not contiguous, pair-count assumption may be off")
    n = len(bin_ids)

    g = df[df["chrom"] == chrom].copy()
    observed_sum = g.groupby("distance")["observed_count"].sum()

    max_d = n - 1
    all_d = np.arange(MIN_DISTANCE_BINS, max_d + 1)
    total_pairs = pd.Series(n - all_d, index=all_d)
    expected = (observed_sum.reindex(all_d, fill_value=0) / total_pairs).rename("expected_count")

    g = g.join(expected, on="distance")
    g["oe"] = (g["observed_count"] / g["expected_count"].replace(0, np.nan)).fillna(0.0)
    return g

## Centrality per chromosome

In [3]:
def chrom_centrality(g, chrom):
    nodes = pd.unique(pd.concat([g["cbin1"], g["cbin2"]]))
    n = len(nodes)
    if n < 3:
        return pd.DataFrame({"bin_id": nodes, "centrality": np.nan}), n, 0

    idx = {b: i for i, b in enumerate(nodes)}
    i_idx = g["cbin1"].map(idx).to_numpy()
    j_idx = g["cbin2"].map(idx).to_numpy()
    w = g["oe"].to_numpy()

    A = sparse.coo_matrix(
        (np.concatenate([w, w]), (np.concatenate([i_idx, j_idx]), np.concatenate([j_idx, i_idx]))),
        shape=(n, n)).tocsr()

    n_comp, labels = connected_components(A, directed=False)
    largest = np.bincount(labels).argmax()
    keep = labels == largest
    n_in_largest = keep.sum()

    centrality = np.full(n, np.nan)
    if n_in_largest >= 3:
        A_sub = A[keep][:, keep]
        try:
            _, eigvec = eigsh(A_sub, k=1, which="LA", maxiter=EIG_MAX_ITER, tol=EIG_TOL)
            centrality[keep] = np.abs(eigvec[:, 0])
        except Exception as e:
            print(f"  [warn] eigsh failed for {chrom} (n={n_in_largest}): {e}, using weighted degree instead")
            centrality[keep] = np.asarray(A_sub.sum(axis=1)).flatten()

    return pd.DataFrame({"bin_id": nodes, "centrality": centrality}), n, n_in_largest

## Run per stage

In [4]:
genes_out = genes[["gene_name", "chrom", "tx_start", "tx_end"]].copy()
component_report = []

for stage in STAGES:
    contact_path = HIC_DIR / f"{stage}_{BIN_SIZE}.n_contact"
    bins_path = HIC_DIR / f"{stage}_{BIN_SIZE}.bins"
    if not (contact_path.exists() and bins_path.exists()):
        print(f"missing files for {stage}")
        genes_out[f"centrality_{stage}"] = np.nan
        continue

    print(f"stage {stage}")
    bins_df = pd.read_csv(bins_path, sep="\t")
    bins_df["chr"] = bins_df["chr"].apply(lambda x: x if str(x).startswith("chr") else f"chr{x}")

    df = pd.read_csv(contact_path, sep="\t", usecols=["cbin1", "cbin2", "observed_count"],
                      dtype={"cbin1": int, "cbin2": int, "observed_count": float})
    df = df[(df["observed_count"] > 0) & (df["cbin1"] != df["cbin2"])].copy()

    bin_chr = bins_df.set_index("cbin")["chr"]
    df["chrom"] = df["cbin1"].map(bin_chr)
    df = df[(df["chrom"] == df["cbin2"].map(bin_chr)) & df["chrom"].isin(MAIN_CHROMS)].copy()
    df["distance"] = (df["cbin1"] - df["cbin2"]).abs()
    df = df[df["distance"] >= MIN_DISTANCE_BINS].copy()

    rows = []
    for chrom in MAIN_CHROMS:
        g = oe_weights(df, bins_df, chrom)
        if len(g) == 0:
            continue
        cent_df, n_total, n_largest = chrom_centrality(g, chrom)
        rows.append(cent_df)
        pct = 100 * (n_total - n_largest) / n_total if n_total else 0
        component_report.append({"stage": stage, "chrom": chrom, "n_bins": n_total,
                                  "n_largest_component": n_largest, "pct_excluded": round(pct, 1)})
        print(f"  {chrom}: {n_total} bins, {n_largest} in largest component ({pct:.1f}% excluded)")

    bin_conn = pd.concat(rows, ignore_index=True)
    bins_df = bins_df.merge(bin_conn, left_on="cbin", right_on="bin_id", how="left")

    merged = pd.merge(genes_out[["gene_name", "chrom", "tx_start", "tx_end"]],
                       bins_df[["chr", "from.coord", "to.coord", "centrality"]],
                       left_on="chrom", right_on="chr", how="inner")
    mapped = merged[(merged["tx_start"] <= merged["to.coord"]) & (merged["tx_end"] >= merged["from.coord"])]
    stage_genes = (mapped.groupby("gene_name")["centrality"].mean().reset_index()
                   .rename(columns={"centrality": f"centrality_{stage}"}))
    genes_out = genes_out.merge(stage_genes, on="gene_name", how="left")

comp_df = pd.DataFrame(component_report)
print("\ncomponent summary:")
print(comp_df.to_string(index=False))

stage C1-8
  chr2L: 2254 bins, 2254 in largest component (0.0% excluded)
  chr2R: 2062 bins, 2062 in largest component (0.0% excluded)
  chr3L: 2411 bins, 2411 in largest component (0.0% excluded)
  chr3R: 2780 bins, 2780 in largest component (0.0% excluded)
  chr4: 121 bins, 121 in largest component (0.0% excluded)
  chrX: 2188 bins, 2188 in largest component (0.0% excluded)
stage C9-13
  chr2L: 2254 bins, 2254 in largest component (0.0% excluded)
  chr2R: 2062 bins, 2062 in largest component (0.0% excluded)
  chr3L: 2411 bins, 2411 in largest component (0.0% excluded)
  chr3R: 2780 bins, 2780 in largest component (0.0% excluded)
  chr4: 121 bins, 121 in largest component (0.0% excluded)
  chrX: 2188 bins, 2188 in largest component (0.0% excluded)
stage C14E
  chr2L: 2254 bins, 2254 in largest component (0.0% excluded)
  chr2R: 2062 bins, 2062 in largest component (0.0% excluded)
  chr3L: 2411 bins, 2411 in largest component (0.0% excluded)
  chr3R: 2780 bins, 2780 in largest componen

## Save

In [5]:
cent_cols = [f"centrality_{s}" for s in STAGES]
n_missing = genes_out[cent_cols].isna().sum()
print("genes with no centrality:")
print(n_missing.to_string())

genes_out.to_csv(OUT, index=False)
print(f"\nsaved -> {OUT.relative_to(ROOT)}, {len(genes_out)} genes")

genes with no centrality:
centrality_C1-8     154
centrality_C9-13    154
centrality_C14E     154
centrality_C14L     154

saved -> data\processed\eigenvector_features.csv, 16846 genes


## Genes with no bin

In [6]:
mapped_genes = set(genes_out.dropna(subset=["centrality_C1-8"])["gene_name"])
unmapped = genes_out[~genes_out["gene_name"].isin(mapped_genes)]
print(unmapped[["gene_name", "chrom", "tx_start", "tx_end"]].head(20))
print(unmapped["chrom"].value_counts())

         gene_name  chrom  tx_start    tx_end
2970       CG18477  chr2L  15661041  15697750
2971       CG18478  chr2L  15670174  15706220
3613       CG31687  chr2L  20450225  20456978
3649       CG31780  chr2L  15661041  15697750
3679       CG31827  chr2L  15670174  15706220
3699       CG31904  chr2L   7734415   7738809
3929       CG32640   chrX  13072038  13072556
3930       CG32641   chrX  13078511  13079029
4840       CG42529  chr3L  19722094  19754917
5338       CG43923  chr2L  15675582  15711099
5339       CG43924  chr2L  15675582  15711099
5519       CG45489   chrX  19822449  19833027
5520       CG45490   chrX  19827574  19838180
5521       CG45491   chrX  19822449  19833027
8255           HIP   chrX   3693183   3695603
8314  His1:CG31617  chr2L  21421855  21537368
8315  His1:CG33801  chr2L  21416953  21537368
8316  His1:CG33804  chr2L  21426916  21537368
8317  His1:CG33807  chr2L  21431963  21537368
8318  His1:CG33810  chr2L  21437011  21482367
chrom
chr2L    136
chrX      13
ch